# Phase 1 – Alzheimer’s Detection Project (Beginner Friendly)
This notebook will guide you step-by-step (like a baby tutorial) through:
- Setting up dataset
- Preprocessing images
- Training a Baseline CNN
- Evaluating results

👉 Run each cell by clicking on it and pressing **Shift + Enter**.
👉 If you see errors, read the red message carefully or ask for help.


## Step 1 – Import Libraries

In [1]:

# Core Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

# Deep Learning Framework
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Evaluation
from sklearn.metrics import classification_report, confusion_matrix
print("✅ Libraries imported successfully")


✅ Libraries imported successfully


## Step 2 – Set Dataset Path
👉 Replace the path below with the folder location where you saved your **Alzheimer_Dataset**.
For example:
- Windows: `C:\\Alzheimer_Dataset`
- Mac/Linux: `/Users/yourname/Alzheimer_Dataset`


In [2]:

# Change this to your local path
src_root = r"C:\Users\Ayush\Alzheimer_Project\Alzheimer_Dataset"  

if os.path.exists(src_root):
    print("✅ Dataset path found:", src_root)
else:
    print("❌ Dataset path not found. Please check again.")


✅ Dataset path found: C:\Users\Ayush\Alzheimer_Project\Alzheimer_Dataset


## Step 3 – Preprocessing
👉 Resize all images to 224x224, normalize pixel values (0–1).
👉 Apply augmentation for training: rotation, zoom, flip.


In [3]:

# Preprocessing & Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    src_root,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
) A

val_gen = train_datagen.flow_from_directory(
    src_root,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)


Found 69151 images belonging to 4 classes.
Found 17286 images belonging to 4 classes.


## Step 4 – Build Baseline CNN

In [4]:

baseline_cnn = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(4, activation='softmax')
])

baseline_cnn.compile(optimizer='adam',
                     loss='categorical_crossentropy',
                     metrics=['accuracy'])

baseline_cnn.summary()


C:\Users\Ayush\anaconda3\envs\alzheimer\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 186624)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    23,888,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,907,908 (91.20 MB)

 Trainable params: 23,907,908 (91.20 MB)

 Non-trainable params: 0 (0.00 B)

## Step 5 – Train the Baseline CNN

In [ ]:

history = baseline_cnn.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15
)  


Epoch 1/15
1844/2161 ━━━━━━━━━━━━━━━━━━━━ 1:15:14 14s/step - accuracy: 0.7758 - loss: 0.6511

## Step 6 – Evaluate the Model

In [ ]:

# Accuracy & Loss Plots
plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.legend()
plt.show()

# Confusion Matrix
val_gen.reset()
preds = baseline_cnn.predict(val_gen)
y_pred = np.argmax(preds, axis=1)
print("Confusion Matrix")
print(confusion_matrix(val_gen.classes, y_pred))
print("Classification Report")
print(classification_report(val_gen.classes, y_pred, target_names=list(val_gen.class_indices.keys())))


# Create models folder if not exists
import os
os.makedirs('models', exist_ok=True)

# Save model
model_path = os.path.join('models', 'baseline_cnn_best.h5')
baseline_cnn.save(model_path)
print(f"✅ Model saved successfully at: {os.path.abspath(model_path)}")
